# Day 2 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B · daily diary on Mentimeter at 15:45**

Bayesian statistics for PhD researchers, Aalborg University 2026. This continues the morning notebook (`day-2-exercises-morning.ipynb`). Select **R (bayes)** and run the cells in order from a fresh kernel. Summaries use dplyr and figures use ggplot2, as this morning.

The data are **synthetic teaching examples**.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| B1 | Counts and exposure | 20 min |
| B2 | Posterior prediction | 25 min |
| B3 | A prior predictive check | 20 min |
| — | Exit response and your group's question set | 15 min |

**Rotate roles:** new coder, checker, two question writers, interviewer and reporter.

**Core:** B1–B3, the exit response and the question set. The optional extension at the end is for spare time only.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## B1 (20 min): counts and exposure

**The setting.** A counter records how many events happen during a time window, like the bikes passing a counter in the lecture. Six windows were watched, but they were **not equally long**:

| window | count (events seen) | exposure (minutes watched) |
|---|---|---|
| 1 | 2 | 1 |
| 2 | 0 | 1 |
| 3 | 4 | 2 |
| 4 | 3 | 1 |
| 5 | 1 | 0.5 |
| 6 | 5 | 2 |

- **Count** yᵢ: the number of events seen in window i. A whole number; 0 is a real observation, not missing data.
- **Exposure** tᵢ: how long window i was watched, in minutes. A longer window gives more chances to see events, so we expect a larger count.
- **Rate** λ: the expected number of events **per minute**. This is the unknown we want to learn. Window i is expected to contain λ × tᵢ events.

**The model.**

- *Likelihood:* yᵢ | λ ~ Poisson(λ tᵢ). We assume the windows are independent and that the rate λ is the same in every window.
- *Prior:* λ ~ Gamma(a = 2, b = 1), with shape a and rate b. Its mean is a/b = 2 events per minute, and it is wide. You can read it as "earlier experience worth about 2 events seen in 1 minute".

**The update (from the lecture).** λ | y ~ Gamma(a + Σyᵢ, b + Σtᵢ): add the **total count** to the shape and the **total exposure** to the rate.

**Tasks.**

1. Compute the total count and the total exposure, then the posterior shape and rate.
2. Compute the posterior mean (shape / rate), the 90% interval (`qgamma`) and P(λ > 2 | y) (`pgamma`).
3. In your own words: what do the two assumptions (independent windows, the same rate in every window) mean here? Give one situation where they could fail.
4. The average of the six counts is 2.5. Why is that **not** an estimate of λ?

*Hint:* give R's gamma functions named arguments, for example `qgamma(0.05, shape = ..., rate = ...)`.

In [ ]:
windows <- tibble(count    = c(2, 0, 4, 3, 1, 5),
                  exposure = c(1, 1, 2, 1, .5, 2))   # minutes observed
a0 <- 2; b0 <- 1                                   # prior Gamma(shape = 2, rate = 1)

# 1. totals:     sum(windows$count), sum(windows$exposure)
# 2. posterior:  shape <- a0 + total count;  rate <- b0 + total exposure
# 3. summaries:  shape / rate,  qgamma(c(0.05, 0.95), shape = shape, rate = rate),
#                1 - pgamma(2, shape = shape, rate = rate)

**Your answer:** write a few sentences here.

## B2 (25 min): posterior prediction

**The question.** How many events will we see in a **new window of 3 minutes**? This is a question about a future count, not about the rate λ. The answer is uncertain for two reasons:

1. we do not know λ exactly: it has the posterior distribution from B1;
2. even if we knew λ, counts vary randomly from window to window (Poisson).

**Simulate in two steps**, as in the lecture: first the rate, then the count.

- Step 1: draw 5,000 rates from the posterior, `rgamma(5000, shape = shape, rate = rate)`.
- Step 2: for each rate, draw one count for 3 minutes, `rpois(5000, lambda = 3 * rate_draws)`.

**Tasks.**

1. Report the mean of the simulated counts, their 90% interval (the 5% and 95% quantiles) and P(new count ≥ 8), which is the share of simulated counts that are 8 or more.
2. Compute the variance of the simulated counts with `var()`. Compare it with **6**, the variance we would get if we pretended λ were exactly its posterior mean 2: a Poisson count has variance = mean = 3 × 2 = 6. Which is bigger, and why?
3. *Check.* The exact distribution of the future count has a name: **negative binomial** (slide "Relationships between distributions: when the parameter itself is uncertain"). In R, `pnbinom(7, size = shape, prob = rate / (rate + 3), lower.tail = FALSE)` gives P(new count ≥ 8). Compare it with your simulated value from task 1.

In [ ]:
set.seed(2026)
t_new <- 3                                               # length of the new window, minutes

# Step 1: rate_draws <- rgamma(5000, shape = shape, rate = rate)
# Step 2: y_new      <- rpois(5000, lambda = t_new * rate_draws)
# Then:   mean(y_new), quantile(y_new, c(0.05, 0.95)), mean(y_new >= 8), var(y_new)

**Your answer:** write a few sentences here.

## B3 (20 min): a prior predictive check

**The idea.** A prior for λ is a statement about the rate, which we never observe directly. What we observe are **counts**. A *prior predictive check* translates the prior into counts: simulate counts using **only the prior** (no data yet), then ask whether these counts look plausible for the setting. If they do not, rethink the prior. It is the same two-step simulation as in B2, with the prior in place of the posterior.

**The setting.** A bike counter on a city cycle path. From earlier experience: about **2 bikes per minute** on average, and a minute with **no bike at all is fairly unusual**, roughly 1 minute in 7.

**Simulate one-minute counts from the B1 prior, Gamma(2, 1).**

- Step 1: draw 5,000 rates from the prior, `rgamma(5000, shape = 2, rate = 1)`.
- Step 2: for each rate, draw one count for a 1-minute window, `rpois(5000, lambda = 1 * rate_draws)`.

**Tasks.**

1. Draw a bar chart of the simulated counts. Report their mean, the share of minutes with no bike, `mean(y_prior == 0)`, and the share with 8 or more bikes.
2. Compare with the experience above. Is the average about right? Is the share of empty minutes about right?
3. Repeat steps 1–2 with the prior **Gamma(20, 10)**. Its mean is also 20 / 10 = 2 bikes per minute, but it counts as "20 bikes seen in 10 minutes" of earlier experience, so it is narrower. Which of the two priors matches the experience better?
4. In one or two sentences: why is it easier to judge a prior by the counts it predicts than by looking at the prior for λ itself?

In [ ]:
set.seed(2026)
# Prior Gamma(2, 1)
# Step 1: rate_draws <- rgamma(5000, shape = 2, rate = 1)
# Step 2: y_prior    <- rpois(5000, lambda = 1 * rate_draws)
# Then:   mean(y_prior), mean(y_prior == 0), mean(y_prior >= 8)
# Bar chart: tibble(count = y_prior) |> ggplot(aes(x = count)) + geom_bar()

# Task 3: the same with rgamma(5000, shape = 20, rate = 10)

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: Compare uncertainty in a rate with uncertainty in a future count. Which process assumption would you test next?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

## Optional extension: a flat prior is flat on one scale only

*Not on the slides; a preview for later days. Skip it if you are short of time.*

Later models often work with the **log-odds** z = log(p / (1 − p)) instead of p. In R: `qlogis(p)` turns p into z and `plogis(z)` turns z back into p. A prior that is flat for p is **not** flat for z.

**Rule for changing scale.** If p has density f(p) and z is a smooth function of p, then the density of z is f(p) × |dp/dz|. Here dp/dz = p(1 − p), and the flat prior has f(p) = 1, so the density of z is **p(1 − p)**, with p = plogis(z).

**Tasks.**

1. Plot the density p(1 − p) against z for z from −8 to 8.
2. Check it by simulation: draw 5,000 values of p from Beta(1, 1), turn them into z with `qlogis()`, and draw a histogram of z.
3. Is the flat prior for p also "uninformative" about z?

In [ ]:
z_grid <- seq(-8, 8, length.out = 500)
# density_z <- plogis(z_grid) * (1 - plogis(z_grid))
# z_sim     <- qlogis(rbeta(5000, 1, 1))

**Your answer:** write a few sentences here.

## Core submission

Complete A1–A3 (morning notebook) and B1–B3, both exit responses, both question sets and your own interpretations. Save outputs and plots. Optional extensions are not required. For absences use the make-up instructions; deadlines and the submission route will be announced. Final assessment is individual and pass/fail.

---
**Everyone back in the room at 15:00 for Peer review B.** Save this notebook (Ctrl/Cmd + S). Daily diary on Mentimeter at 15:45.